# Example 1 — Gaussian terminal condition in $d=2$: detailed house $\to$ flowers

This notebook keeps the **Gaussian terminal condition** setup from Section 3.3 / **Algorithm 1** of the manuscript, but replaces the tiny 12-point toy geometry with **dense procedural point clouds**.

Here:
- the initial labeled atomic configuration is a more detailed **house**
- the terminal target configuration is a small **flower bed**
- the example uses **180 particles** (well above 100)
- particle **mass is frozen** and shown by color
- the main output is an **interactive time slider** (with play/pause controls)

Because many more particles imply much smaller individual masses, the horizon is reduced accordingly so the diffusion remains visually sharp. The animation therefore shows both **normalized time** $t/T$ and the actual time $t$ in scientific notation.

In [ ]:
import numpy as np
import sys
from pathlib import Path
from scipy.optimize import linear_sum_assignment


ROOT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents, Path("/mnt/data").resolve())
ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "wasserstein_conditioning_algorithms.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate wasserstein_conditioning_algorithms.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.wasserstein_conditioning_algorithms import (
    shortest_periodic_displacement,
    simulate_gaussian_terminal_em,
)

np.set_printoptions(precision=3, suppress=True)


In [ ]:
import plotly.graph_objects as go

from notebooks.support import (
    center_trace,
    circle_trace,
    configure_plotly,
    line_trace as _line_trace,
    make_particle_animation as _make_particle_animation,
)

configure_plotly()


def line_trace(points, name, color="rgba(80,80,80,0.55)", dash="dot", close=False, showlegend=True, marker_size=4):
    return _line_trace(
        points,
        name,
        color=color,
        dash=dash,
        close=close,
        showlegend=showlegend,
        marker_size=marker_size,
        mode="lines+markers",
    )



def _frame_subtitle(t, horizon):
    if horizon <= 0.0:
        return f"time = {t:.3f}"
    if horizon < 1e-2:
        return f"t/T = {t / horizon:.3f},  t = {t:.2e}"
    return f"time = {t:.3f}"



def make_particle_animation(
    positions,
    times,
    masses,
    title,
    static_traces=None,
    marker_size=8,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
):
    return _make_particle_animation(
        positions,
        times,
        masses,
        title,
        static_traces=static_traces,
        marker_size=marker_size,
        x_range=x_range,
        y_range=y_range,
        mass_format=".5f",
        time_formatter=_frame_subtitle,
        slider_label_formatter=lambda t, h: f"{(float(t) / h if h > 0 else 0.0):.2f}",
        currentvalue_prefix="t/T = ",
        width=820,
        height=720,
        play_frame_duration=110,
    )


In [ ]:
# --- Dense procedural house and flower-bed point sets on the flat torus [0, 1)^2 ---

def sample_segment(p0, p1, n, *, endpoint=True):
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    t = np.linspace(0.0, 1.0, n, endpoint=endpoint)[:, None]
    return p0 + (p1 - p0) * t

def rectangle_outline(x0, y0, x1, y1, n_edge):
    return np.vstack([
        sample_segment((x0, y0), (x1, y0), n_edge, endpoint=False),
        sample_segment((x1, y0), (x1, y1), n_edge, endpoint=False),
        sample_segment((x1, y1), (x0, y1), n_edge, endpoint=False),
        sample_segment((x0, y1), (x0, y0), n_edge, endpoint=False),
    ])

def rectangle_fill(x0, y0, x1, y1, nx, ny):
    xs = np.linspace(x0, x1, nx)
    ys = np.linspace(y0, y1, ny)
    X, Y = np.meshgrid(xs, ys)
    return np.column_stack([X.ravel(), Y.ravel()])

def triangle_fill(a, b, c, levels):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    c = np.asarray(c, dtype=float)
    pts = []
    for i in range(levels + 1):
        for j in range(levels + 1 - i):
            k = levels - i - j
            w = np.array([i, j, k], dtype=float) / levels
            pts.append(w[0] * a + w[1] * b + w[2] * c)
    return np.asarray(pts)

def ellipse_outline(center, rx, ry, n, rotation=0.0):
    center = np.asarray(center, dtype=float)
    theta = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    xy = np.column_stack([rx * np.cos(theta), ry * np.sin(theta)])
    R = np.array([
        [np.cos(rotation), -np.sin(rotation)],
        [np.sin(rotation),  np.cos(rotation)],
    ])
    return center + xy @ R.T

def ellipse_fill(center, rx, ry, nr, ntheta, rotation=0.0):
    center = np.asarray(center, dtype=float)
    radii = np.sqrt(np.linspace(0.02, 1.0, nr))
    theta = np.linspace(0.0, 2.0 * np.pi, ntheta, endpoint=False)
    R = np.array([
        [np.cos(rotation), -np.sin(rotation)],
        [np.sin(rotation),  np.cos(rotation)],
    ])
    pts = []
    for j, r in enumerate(radii):
        theta_shift = (j % 2) * (np.pi / ntheta / 2.0)
        xy = np.column_stack([rx * r * np.cos(theta + theta_shift), ry * r * np.sin(theta + theta_shift)])
        pts.append(center + xy @ R.T)
    pts.append(center[None, :])
    return np.vstack(pts)

def unique_points(points, decimals=6):
    pts = np.asarray(points, dtype=float)
    rounded = np.round(pts, decimals=decimals)
    _, idx = np.unique(rounded, axis=0, return_index=True)
    return pts[np.sort(idx)]

def farthest_point_sample(points, n_points):
    pts = np.asarray(points, dtype=float)
    if n_points >= len(pts):
        return pts.copy()

    centroid = np.mean(pts, axis=0)
    first = int(np.argmax(np.sum((pts - centroid) ** 2, axis=1)))
    chosen = [first]
    min_sqdist = np.sum((pts - pts[first]) ** 2, axis=1)

    for _ in range(1, n_points):
        idx = int(np.argmax(min_sqdist))
        chosen.append(idx)
        min_sqdist = np.minimum(min_sqdist, np.sum((pts - pts[idx]) ** 2, axis=1))

    return pts[np.asarray(chosen)]

def make_house_candidates():
    body_outline = rectangle_outline(0.18, 0.18, 0.82, 0.58, 50)
    roof_left = sample_segment((0.18, 0.58), (0.50, 0.84), 40, endpoint=False)
    roof_right = sample_segment((0.50, 0.84), (0.82, 0.58), 40, endpoint=False)
    roof_base = sample_segment((0.22, 0.58), (0.78, 0.58), 35, endpoint=False)

    door_outline = rectangle_outline(0.43, 0.18, 0.57, 0.42, 22)
    door_arch = np.column_stack([
        np.linspace(0.43, 0.57, 6),
        0.42 + 0.03 * np.sin(np.linspace(np.pi, 0.0, 6)),
    ])

    window_left = rectangle_outline(0.26, 0.33, 0.38, 0.46, 16)
    window_right = rectangle_outline(0.62, 0.33, 0.74, 0.46, 16)
    window_cross = np.vstack([
        sample_segment((0.32, 0.33), (0.32, 0.46), 12, endpoint=False),
        sample_segment((0.26, 0.395), (0.38, 0.395), 12, endpoint=False),
        sample_segment((0.68, 0.33), (0.68, 0.46), 12, endpoint=False),
        sample_segment((0.62, 0.395), (0.74, 0.395), 12, endpoint=False),
    ])

    chimney = np.vstack([
        rectangle_outline(0.63, 0.62, 0.71, 0.78, 14),
        sample_segment((0.63, 0.62), (0.50, 0.72), 12, endpoint=False),
        sample_segment((0.71, 0.62), (0.58, 0.75), 12, endpoint=False),
    ])

    body_fill = rectangle_fill(0.20, 0.20, 0.80, 0.56, 19, 9)
    door_or_window = (
        ((body_fill[:, 0] > 0.41) & (body_fill[:, 0] < 0.59) & (body_fill[:, 1] < 0.43)) |
        ((body_fill[:, 0] > 0.25) & (body_fill[:, 0] < 0.39) & (body_fill[:, 1] > 0.32) & (body_fill[:, 1] < 0.47)) |
        ((body_fill[:, 0] > 0.61) & (body_fill[:, 0] < 0.75) & (body_fill[:, 1] > 0.32) & (body_fill[:, 1] < 0.47))
    )
    body_fill = body_fill[~door_or_window]

    roof_fill = triangle_fill((0.20, 0.58), (0.80, 0.58), (0.50, 0.82), 22)
    path_fill = rectangle_fill(0.46, 0.06, 0.54, 0.18, 3, 6)

    return unique_points(np.vstack([
        body_outline,
        roof_left,
        roof_right,
        roof_base,
        door_outline,
        door_arch,
        window_left,
        window_right,
        window_cross,
        chimney,
        body_fill,
        roof_fill,
        path_fill,
    ]))

def make_flower_candidates():
    pts = []

    def add_flower(center, *, size=1.0, petals=8, rotation=0.0):
        pts.append(ellipse_fill(center, 0.035 * size, 0.035 * size, nr=5, ntheta=10))
        for k in range(petals):
            angle = rotation + 2.0 * np.pi * k / petals
            petal_center = np.asarray(center) + 0.065 * size * np.array([np.cos(angle), np.sin(angle)])
            pts.append(ellipse_outline(petal_center, 0.060 * size, 0.022 * size, 36, rotation=angle))
            pts.append(ellipse_fill(petal_center, 0.050 * size, 0.018 * size, nr=4, ntheta=8, rotation=angle))
        pts.append(ellipse_outline(center, 0.12 * size, 0.12 * size, 24))

    add_flower((0.50, 0.71), size=1.00, petals=8, rotation=np.pi / 8.0)
    add_flower((0.32, 0.63), size=0.78, petals=7, rotation=np.pi / 14.0)
    add_flower((0.68, 0.63), size=0.78, petals=7, rotation=-np.pi / 14.0)

    pts.append(sample_segment((0.50, 0.60), (0.50, 0.18), 70, endpoint=False))
    pts.append(sample_segment((0.32, 0.54), (0.38, 0.18), 58, endpoint=False))
    pts.append(sample_segment((0.68, 0.54), (0.62, 0.18), 58, endpoint=False))

    for center, rotation, scale in [
        ((0.44, 0.33), -0.70, 1.00),
        ((0.56, 0.31),  0.70, 1.00),
        ((0.28, 0.30), -0.90, 0.70),
        ((0.72, 0.30),  0.90, 0.70),
    ]:
        pts.append(ellipse_outline(center, 0.070 * scale, 0.030 * scale, 30, rotation=rotation))
        pts.append(ellipse_fill(center, 0.060 * scale, 0.025 * scale, nr=4, ntheta=8, rotation=rotation))
        c = np.asarray(center)
        direction = np.array([np.cos(rotation), np.sin(rotation)]) * 0.065 * scale
        pts.append(sample_segment(c - 0.8 * direction, c + 0.8 * direction, 12, endpoint=False))

    grass_x = np.linspace(0.18, 0.82, 28)
    grass_y = 0.14 + 0.015 * np.sin(np.linspace(0.0, 3.0 * np.pi, len(grass_x)))
    pts.append(np.column_stack([grass_x, grass_y]))

    return unique_points(np.vstack(pts))

# Dense point clouds.
n_particles = 180
house_candidates = make_house_candidates()
flower_candidates = make_flower_candidates()

house_points = farthest_point_sample(house_candidates, n_particles)
flower_points = farthest_point_sample(flower_candidates, n_particles)

# Reference outlines used only for plotting.
house_outline = np.array([
    [0.18, 0.18],
    [0.82, 0.18],
    [0.82, 0.58],
    [0.50, 0.84],
    [0.18, 0.58],
], dtype=float)
house_door = np.array([
    [0.43, 0.18],
    [0.43, 0.42],
    [0.57, 0.42],
    [0.57, 0.18],
], dtype=float)
house_window_left = np.array([
    [0.26, 0.33],
    [0.38, 0.33],
    [0.38, 0.46],
    [0.26, 0.46],
], dtype=float)
house_window_right = np.array([
    [0.62, 0.33],
    [0.74, 0.33],
    [0.74, 0.46],
    [0.62, 0.46],
], dtype=float)
house_chimney = np.array([
    [0.63, 0.62],
    [0.63, 0.78],
    [0.71, 0.78],
    [0.71, 0.62],
], dtype=float)

flower_main_outline = ellipse_outline((0.50, 0.71), 0.16, 0.11, 160)
flower_left_outline = ellipse_outline((0.32, 0.63), 0.13, 0.09, 140)
flower_right_outline = ellipse_outline((0.68, 0.63), 0.13, 0.09, 140)
flower_centers = np.array([
    [0.50, 0.71],
    [0.32, 0.63],
    [0.68, 0.63],
], dtype=float)
flower_stem_center = np.array([[0.50, 0.60], [0.50, 0.18]], dtype=float)
flower_stem_left = np.array([[0.32, 0.54], [0.38, 0.18]], dtype=float)
flower_stem_right = np.array([[0.68, 0.54], [0.62, 0.18]], dtype=float)
flower_leaf_left = ellipse_outline((0.44, 0.33), 0.07, 0.03, 100, rotation=-0.70)
flower_leaf_right = ellipse_outline((0.56, 0.31), 0.07, 0.03, 100, rotation=0.70)

# Smooth near-uniform masses: enough variation for the colorbar, but not enough
# to make tiny particles numerically unstable.
raw_masses = 1.0 + 0.08 * np.sin(2.0 * np.pi * house_points[:, 0]) + 0.06 * np.cos(2.0 * np.pi * house_points[:, 1])
raw_masses = np.clip(raw_masses, 0.85, None)
masses = raw_masses / raw_masses.sum()

# Because the Gaussian target is label-aware, assign house points to flower points
# using a minimal Euclidean pairing before simulating.
cost_matrix = np.sum((house_points[:, None, :] - flower_points[None, :, :]) ** 2, axis=-1)
row_ind, col_ind = linear_sum_assignment(cost_matrix)

target_positions = np.empty_like(flower_points)
target_positions[row_ind] = flower_points[col_ind]

# Simulation parameters. With ~180 particles, each mass is smaller than in the
# 12-particle toy example, so we shorten the horizon to keep the motion crisp.
lambda_ = 5_000_000
horizon = 4.0e-05
step_size = horizon / 240
seed = 33

initial_periodic_distance = np.sqrt(np.sum(shortest_periodic_displacement(house_points, target_positions) ** 2, axis=-1)) @ masses

print("number of particles:", len(masses))
print("mass range:", (float(masses.min()), float(masses.max())))
print("house candidates -> selected:", len(house_candidates), "->", len(house_points))
print("flower candidates -> selected:", len(flower_candidates), "->", len(flower_points))
print("initial weighted mean periodic distance:", float(initial_periodic_distance))
print("horizon:", horizon, "step_size:", step_size, "seed:", seed)

In [ ]:

rng = np.random.default_rng(seed)

sim = simulate_gaussian_terminal_em(
    masses=masses,
    target_positions=target_positions,
    lambda_=lambda_,
    horizon=horizon,
    step_size=step_size,
    initial_positions=house_points,
    drift_mode="wrapped",
    image_radius=1,
    rng=rng,
    store_drifts=True,
)

print("positions array shape:", sim.positions.shape)
print("final time:", float(sim.times[-1]))


In [ ]:
static_traces = [
    line_trace(house_outline, name="house outline", color="rgba(30, 144, 255, 0.55)", dash="dash", close=True, marker_size=3),
    line_trace(house_door, name="house door", color="rgba(30, 144, 255, 0.45)", dash="dash", close=True, showlegend=False, marker_size=3),
    line_trace(house_window_left, name="house window", color="rgba(30, 144, 255, 0.40)", dash="dash", close=True, showlegend=False, marker_size=3),
    line_trace(house_window_right, name="house window", color="rgba(30, 144, 255, 0.40)", dash="dash", close=True, showlegend=False, marker_size=3),
    line_trace(house_chimney, name="house chimney", color="rgba(30, 144, 255, 0.40)", dash="dash", close=True, showlegend=False, marker_size=3),

    line_trace(flower_main_outline, name="main flower", color="rgba(220, 20, 60, 0.55)", dash="dot", close=True, marker_size=2),
    line_trace(flower_left_outline, name="side flowers", color="rgba(220, 20, 60, 0.45)", dash="dot", close=True, showlegend=False, marker_size=2),
    line_trace(flower_right_outline, name="side flowers", color="rgba(220, 20, 60, 0.45)", dash="dot", close=True, showlegend=False, marker_size=2),
    center_trace(flower_centers, name="flower centers", color="rgba(220, 20, 60, 0.75)", symbol="x", size=9, showlegend=False),
    line_trace(flower_stem_center, name="stems", color="rgba(34, 139, 34, 0.55)", dash="dot", close=False, showlegend=False, marker_size=3),
    line_trace(flower_stem_left, name="stems", color="rgba(34, 139, 34, 0.55)", dash="dot", close=False, showlegend=False, marker_size=3),
    line_trace(flower_stem_right, name="stems", color="rgba(34, 139, 34, 0.55)", dash="dot", close=False, showlegend=False, marker_size=3),
    line_trace(flower_leaf_left, name="leaves", color="rgba(34, 139, 34, 0.50)", dash="dot", close=True, showlegend=False, marker_size=2),
    line_trace(flower_leaf_right, name="leaves", color="rgba(34, 139, 34, 0.50)", dash="dot", close=True, showlegend=False, marker_size=2),
]

fig = make_particle_animation(
    positions=sim.positions,
    times=sim.times,
    masses=sim.masses,
    title="Example 1: Gaussian terminal condition (detailed house → flowers)",
    static_traces=static_traces,
    marker_size=8,
)
fig.show()

In [ ]:
# Weighted mean periodic distance to the assigned flower targets.
periodic_distances = np.sqrt(
    np.sum(
        shortest_periodic_displacement(sim.positions, target_positions[None, :, :]) ** 2,
        axis=-1,
    )
)
weighted_mean_distance = periodic_distances @ sim.masses

import plotly.graph_objects as go

distance_fig = go.Figure()
distance_fig.add_trace(
    go.Scatter(
        x=sim.times,
        y=weighted_mean_distance,
        mode="lines",
        name="weighted mean periodic distance",
    )
)
distance_fig.update_layout(
    title="How close the particles stay to their assigned flower targets",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="weighted mean periodic distance",
)
distance_fig.show()

print("initial weighted mean periodic distance:", float(weighted_mean_distance[0]))
print("final weighted mean periodic distance:", float(weighted_mean_distance[-1]))